### Configuration of model 

In [ ]:
config = {
    "vocab_size": 10000,
    "max_seq_len": 512,
    "qkv_bias": False,
    "d_model": 384,
    "n_layers": 12,
    "n_heads": 8,
    "d_ff": 1536,
    "dropout": 0.1,
}

#### Trainging Configs

In [ ]:
training_config = {
    "batch_size": 32,
    "grad_accum_steps": 1,
    "max_lr": 6e-4,
    "min_lr": 6e-5,
    "warmup_steps": 2000,

    "weight_decay": 0.01,
    "betas": (0.9, 0.95),
    "adam_eps": 1e-8,
    "grad_clip": 1.0,

    "log_interval": 50,
    "eval_interval": 1000,
    "eval_batches": 40,
    "save_interval": 1000,
    "archive_interval": 5000,

    "target_tokens": 550_000_000,
    "use_torch_compile": True,
    "precision": precision,
    "max_consecutive_nonfinite_skips": 20,
    "amp_resume_scale_cap": 65536.0,
}

#### Imports

In [ ]:
import os, gc, json, math, time, random, shutil
from contextlib import nullcontext
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

### MultiHead Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads 

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out) 
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) 
        queries = self.W_query(x)
        values = self.W_value(x)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

## GPT Implementation

### LyaerNormalization

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

### Gelu Activation

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

### Feed Forward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["d_model"], 4 * cfg["d_model"]),
            GELU(),
            nn.Linear(4 * cfg["d_model"], cfg["d_model"]),
        )

    def forward(self, x):
        return self.layers(x)

### Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["d_model"],
            d_out=cfg["d_model"],
            context_length=cfg["max_seq_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["dropout"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["d_model"])
        self.norm2 = LayerNorm(cfg["d_model"])
        self.drop_shortcut = nn.Dropout(cfg["dropout"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  
        x = self.drop_shortcut(x)
        x = x + shortcut

        
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut 

        return x

### Final Transformer

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(
            cfg["vocab_size"], cfg["emb_dim"]
        )

        self.pos_emb = nn.Embedding(
            cfg["context_length"], cfg["emb_dim"]
        )

        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])

        self.out_head = self.tok_emb

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        x = tok_embeds + pos_embeds

        x = self.drop_emb(x)

        for block in self.trf_blocks:
            x = block(x)

        x = self.final_norm(x)

        logits = self.out_head(x)

        return logits

In [ ]:
model = GPTModel(config)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

### Training the Gpt Trnsformer 

### Load the tokenizer 

In [ ]:
sp = spm.SentencePieceProcessor(model_file=tokenizer_model_path)
actual_vocab_size = sp.get_piece_size()

print("Tokenizer vocabulary:", actual_vocab_size)
print("Model vocabulary:", config["vocab_size"])

### Read the token ids stored in bin file 

In [ ]:
class PackedMemmapTokenDataset(Dataset):

    def __init__(self, bin_path, seq_len):
        self.seq_len = seq_len
        self.data = np.memmap(bin_path, dtype=np.uint16, mode="r")

    def __len__(self):
        return (len(self.data) - 1) // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len
        tokens = self.data[start : start + self.seq_len + 1].copy()

        return torch.tensor(tokens, dtype=torch.long)

In [ ]:

train_dataset = PackedMemmapTokenDataset(
    train_bin_path,
    config["max_seq_len"]
)

total_tokens = len(train_dataset.data)

batch_size = training_config["batch_size"]
grad_accum_steps = training_config["grad_accum_steps"]

effective_batch_sequences = batch_size * grad_accum_steps

tokens_per_step = (
    effective_batch_sequences * config["max_seq_len"]
)

max_steps = math.ceil(
    training_config["target_tokens"] / tokens_per_step
)

training_config["max_steps"] = max_steps

print("Total tokens:", total_tokens)
print("Effective sequences per update:", effective_batch_sequences)
print("Tokens per update:", tokens_per_step)
print("Max optimizer steps:", max_steps)

sample = train_dataset[0]

assert sample.dtype == torch.uint16
assert len(sample) == config["max_seq_len"] + 1

#### Creating DatlOADER 

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=training_config["batch_size"],
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=training_config["batch_size"],
    shuffle=False,
    drop_last=True
)


### Configure the Optimizer 

In [ ]:
def configure_optimizer(model, cfg):

    decay_params = []
    no_decay_params = []

   
    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        
        if param.dim() >= 2:
            decay_params.append(param)

    
        else:
            no_decay_params.append(param)

    optimizer = AdamW(
        [
            {
                "params": decay_params,
                "weight_decay": cfg["weight_decay"]
            },
            {
                "params": no_decay_params,
                "weight_decay": 0.0
            }
        ],
        lr=cfg["max_lr"],
        betas=tuple(cfg["betas"]),
        eps=cfg["adam_eps"]
    )

    return optimizer



### Learning rate Scheduler 

In [ ]:
class WarmupCosineScheduler:

    def __init__(self, optimizer, warmup_steps, max_steps, max_lr, min_lr):

        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.max_lr = max_lr
        self.min_lr = min_lr

    def get_lr(self, step):

        # Warmup: slowly increase learning rate
        if step < self.warmup_steps:
            lr = self.max_lr * (step + 1) / self.warmup_steps

        # Cosine decay: slowly decrease learning rate
        else:
            ratio = (step - self.warmup_steps) / (
                self.max_steps - self.warmup_steps
            )

            ratio = min(ratio, 1.0)

            lr = self.min_lr + 0.5 * (
                1 + math.cos(math.pi * ratio)
            ) * (self.max_lr - self.min_lr)

        return lr

    def step(self, step):

        lr = self.get_lr(step)

        for group in self.optimizer.param_groups:
            group["lr"] = lr

        return lr



#### save and load the chekcpoint 

In [ ]:
def save_checkpoint(
    path, model, optimizer, scheduler, step, loss
):

    checkpoint = {
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "loss": loss,
    }

    torch.save(checkpoint, path)

    print("Checkpoint saved:", path)


def load_checkpoint(
    path, model, optimizer, scheduler
):

    if not os.path.exists(path):
        print("No checkpoint found.")
        return 0

    checkpoint = torch.load(
        path,
        map_location="cpu"
    )

    # Restore model
    model.load_state_dict(
        checkpoint["model"]
    )

    # Restore optimizer
    optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    # Restore scheduler
    scheduler.load_state_dict(
        checkpoint["scheduler"]
    )

    step = checkpoint["step"]
    loss = checkpoint["loss"]

    print("Checkpoint loaded:", path)
    print("Resuming from step:", step)

    return step, loss



### Loadidn preexisiting checkpoint if it exisits or else starting from 0 

In [ ]:
base_model = Transformer(config).to(device)

optimizer = configure_optimizer(
    base_model,
    training_config
)

scheduler = WarmupCosineScheduler(
    optimizer,
    training_config["warmup_steps"],
    max_steps,
    training_config["max_lr"],
    training_config["min_lr"]
)

if os.path.exists(latest_ckpt_path):

    print("Checkpoint found. Loading...")

    current_step, last_loss = load_checkpoint(
        latest_ckpt_path,
        base_model,
        optimizer,
        scheduler
    )

    print(f"Resuming from step: {current_step:,}")

else:

    print("No checkpoint found. Starting from step 0.")

    current_step = 0
    last_loss = None


scheduler.step(current_step)
model = base_model

print(f"Starting step: {current_step:,}")
print(f"Target final step: {max_steps:,}")
print(
    f"Remaining steps: {max_steps - current_step:,}"
)


### Testing on held out data 

In [ ]:
def packed_to_xy(packed):

    packed = packed.to(device).long()

    x = packed[:, :-1]
    y = packed[:, 1:]

    return x, y


@torch.inference_mode()
def evaluate_validation(model, dataloader, max_batches=40):

    model.eval()

    total_loss = 0
    total_tokens = 0

    for i, packed in enumerate(dataloader):

        x, y = packed_to_xy(packed)

        logits = model(x)

        loss = F.cross_entropy(
            logits.reshape(-1, config["vocab_size"]),
            y.reshape(-1),
            reduction="sum"
        )

        total_loss += loss.item()
        total_tokens += y.numel()

        if i + 1 >= max_batches:
            break

    loss = total_loss / total_tokens
    perplexity = math.exp(loss)

    return loss, perplexity



### Traiinging Loop 

In [ ]:
def train_model(
    model,
    optimizer,
    scheduler,
    train_loader,
    val_loader,
    start_step,
    max_steps
):

    step = start_step
    train_iter = iter(train_loader)

    while step < max_steps:

        model.train()

        # Set learning rate
        lr = scheduler.step(step)

        # Reset gradients
        optimizer.zero_grad()

        total_loss = 0

        # Gradient accumulation
        for _ in range(training_config["grad_accum_steps"]):

            # Get batch
            try:
                packed = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                packed = next(train_iter)

            # Create input and target
            x, y = packed_to_xy(packed)

            # Forward pass
            logits = model(x)

            # Calculate loss
            loss = F.cross_entropy(
                logits.reshape(-1, config["vocab_size"]),
                y.reshape(-1)
            )

            # Backpropagation
            loss = loss / training_config["grad_accum_steps"]
            loss.backward()

            total_loss += loss.item()

        # Update model weights
        optimizer.step()

        step += 1

        # Print training progress
        if step % training_config["log_interval"] == 0:

            avg_loss = total_loss / training_config["grad_accum_steps"]

            print(
                f"Step: {step:,} | "
                f"Loss: {avg_loss:.4f} | "
                f"LR: {lr:.2e}"
            )

        # Validation
        if step % training_config["eval_interval"] == 0:

            val_loss, val_ppl = evaluate_validation(
                model,
                val_loader,
                max_batches=training_config["eval_batches"]
            )

            print(
                f"Validation | "
                f"Loss: {val_loss:.4f} | "
                f"PPL: {val_ppl:.2f}"
            )

        # Save checkpoint
        if step % training_config["save_interval"] == 0:

            save_checkpoint(
                latest_ckpt_path,
                model,
                optimizer,
                scheduler,
                step,
                total_loss
            )

            print("Checkpoint saved.")

    return step


In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    current_step, history = train_phase2_fast(
        model=model,
        base_model=base_model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        train_loader=train_loader,
        val_loader=val_loader,
        start_step=current_step,
        history=history,
    )

    if current_step != max_steps:
        raise RuntimeError(
            f"Training returned at step {current_step:,}, expected {max_steps:,}."
        )

    print(f"Training complete at step {current_step:,}")
else:
    print("Training skipped. Set RUN_TRAINING=True to resume.")


### PLoting traingin/loss curves 

In [ ]:
def plot_training_curves(history):
    plt.figure(figsize=(10, 5))

    if history.get("train"):
        plt.plot(
            [r["step"] for r in history["train"]],
            [r["train_loss"] for r in history["train"]],
            label="Training loss",
        )

    if history.get("val"):
        plt.plot(
            [r["step"] for r in history["val"]],
            [r["val_loss"] for r in history["val"]],
            marker="o",
            label="Validation loss",
        )

    plt.title("Telugu Model H — Phase 2 loss curves")
    plt.xlabel("Optimizer step")
    plt.ylabel("Cross-entropy loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_training_curves(history)

### Evaluate the model on cross entropy and perplexity 

In [ ]:
@torch.inference_mode()
def evaluate_lm_metrics(eval_model, dataloader, tokenizer, max_batches=100):
    was_training = eval_model.training
    eval_model.eval()

    total_nll_nats = 0.0
    total_tokens = 0
    total_bytes = 0

    for batch_idx, packed in enumerate(dataloader):
        inputs, targets = packed_to_xy(packed)

        with autocast_context():
            logits = eval_model(inputs)
            nll = F.cross_entropy(
                logits.reshape(-1, config["vocab_size"]),
                targets.reshape(-1),
                reduction="sum",
            )

        total_nll_nats += float(nll.item())
        total_tokens += targets.numel()

        for seq in targets.detach().cpu().tolist():
            text = tokenizer.decode(seq)
            total_bytes += len(text.encode("utf-8"))

        if max_batches is not None and batch_idx + 1 >= max_batches:
            break

    ce = total_nll_nats / max(total_tokens, 1)
    ppl = math.exp(min(ce, 20.0))
    bpb = total_nll_nats / (math.log(2.0) * max(total_bytes, 1))

    if was_training:
        eval_model.train()

    return {
        "cross_entropy": ce,
        "perplexity": ppl,
        "bits_per_byte": bpb,
        "evaluated_tokens": total_tokens,
        "evaluated_utf8_bytes": total_bytes,
    }

lm_metrics = evaluate_lm_metrics(base_model, val_loader, sp, max_batches=100)
display(pd.DataFrame([lm_metrics]))

#### Greedy decoding and temperatures 0.5, 1.0 and 1.5 are evaluated against held-out continuations using BLEU-4, chrF, ROUGE-L, Distinct-1/2 and repeated 3-gram rate.

In [ ]:
@torch.inference_mode()
def generate_batch(
    eval_model,
    prefix_ids,
    max_new_tokens,
    temperature=None,
    sample=False,
    seed=SEED,
):
    was_training = eval_model.training
    eval_model.eval()

    if sample:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    ids = prefix_ids.to(device, dtype=torch.long)

    for _ in range(max_new_tokens):
        context = ids[:, -config["max_seq_len"] :]

        with autocast_context():
            logits = eval_model(context)[:, -1, :]

        if sample:
            temp = float(temperature)
            probs = F.softmax(logits.float() / temp, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
        else:
            next_id = torch.argmax(logits, dim=-1, keepdim=True)

        ids = torch.cat([ids, next_id], dim=1)

    if was_training:
        eval_model.train()
    return ids


def get_generation_eval_batch(
    dataset, n_examples=16, prefix_len=128, continuation_len=64
):
    prefixes, references = [], []

    for i in range(n_examples):
        packed = dataset[i].long()
        prefixes.append(packed[:prefix_len])
        references.append(
            packed[prefix_len : prefix_len + continuation_len]
        )

    return torch.stack(prefixes), torch.stack(references)


In [ ]:
def _ngrams(seq, n):
    return [tuple(seq[i:i+n]) for i in range(len(seq) - n + 1)]


def corpus_bleu4(reference_texts, hypothesis_texts):
    clipped = [0] * 4
    totals = [0] * 4
    ref_len = 0
    hyp_len = 0

    for ref_text, hyp_text in zip(reference_texts, hypothesis_texts):
        ref = ref_text.split()
        hyp = hyp_text.split()
        ref_len += len(ref)
        hyp_len += len(hyp)

        for n in range(1, 5):
            ref_counts = Counter(_ngrams(ref, n))
            hyp_counts = Counter(_ngrams(hyp, n))
            totals[n-1] += sum(hyp_counts.values())
            clipped[n-1] += sum(
                min(count, ref_counts[gram])
                for gram, count in hyp_counts.items()
            )

    if hyp_len == 0:
        return 0.0

    log_precision = 0.0
    for c, t in zip(clipped, totals):
        log_precision += 0.25 * math.log((c + 1.0) / (t + 1.0))

    bp = (
        1.0 if hyp_len > ref_len
        else math.exp(1.0 - ref_len / max(hyp_len, 1))
    )
    return 100.0 * bp * math.exp(log_precision)


def corpus_chrf(reference_texts, hypothesis_texts, max_n=6, beta=2.0):
    precisions, recalls = [], []

    for n in range(1, max_n + 1):
        match = hyp_total = ref_total = 0

        for ref_text, hyp_text in zip(reference_texts, hypothesis_texts):
            ref_chars = list("".join(ref_text.split()))
            hyp_chars = list("".join(hyp_text.split()))
            ref_counts = Counter(_ngrams(ref_chars, n))
            hyp_counts = Counter(_ngrams(hyp_chars, n))

            match += sum(
                min(count, ref_counts[gram])
                for gram, count in hyp_counts.items()
            )
            hyp_total += sum(hyp_counts.values())
            ref_total += sum(ref_counts.values())

        precisions.append(match / max(hyp_total, 1))
        recalls.append(match / max(ref_total, 1))

    p = sum(precisions) / len(precisions)
    r = sum(recalls) / len(recalls)
    beta2 = beta * beta

    return 100.0 * (1 + beta2) * p * r / max(beta2 * p + r, 1e-12)


def _lcs_length(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = [0] * (len(b) + 1)

    for token_a in a:
        cur = [0]
        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                cur.append(prev[j-1] + 1)
            else:
                cur.append(max(cur[-1], prev[j]))
        prev = cur

    return prev[-1]


def rouge_l_f1(reference_texts, hypothesis_texts):
    scores = []

    for ref_text, hyp_text in zip(reference_texts, hypothesis_texts):
        ref = ref_text.split()
        hyp = hyp_text.split()

        if not ref or not hyp:
            scores.append(0.0)
            continue

        lcs = _lcs_length(ref, hyp)
        p = lcs / len(hyp)
        r = lcs / len(ref)
        scores.append(2 * p * r / max(p + r, 1e-12))

    return 100.0 * sum(scores) / max(len(scores), 1)


def distinct_n(token_sequences, n):
    grams = []
    for seq in token_sequences:
        grams.extend(_ngrams(seq, n))
    return len(set(grams)) / max(len(grams), 1)


def repetition_rate(token_sequences, n=3):
    grams = []
    for seq in token_sequences:
        grams.extend(_ngrams(seq, n))
    return 1.0 - len(set(grams)) / max(len(grams), 1)

In [ ]:
GEN_EXAMPLES = 16
PREFIX_LEN = 128
CONTINUATION_LEN = 64

prefixes, reference_ids = get_generation_eval_batch(
    val_dataset,
    n_examples=GEN_EXAMPLES,
    prefix_len=PREFIX_LEN,
    continuation_len=CONTINUATION_LEN,
)

strategies = [
    ("greedy", None, False),
    ("temperature_0.5", 0.5, True),
    ("temperature_1.0", 1.0, True),
    ("temperature_1.5", 1.5, True),
]

reference_texts = [sp.decode(seq.tolist()) for seq in reference_ids]
rows = []
generated_examples = {}

for name, temperature, sample in strategies:
    full = generate_batch(
        base_model,
        prefixes,
        max_new_tokens=CONTINUATION_LEN,
        temperature=temperature,
        sample=sample,
        seed=SEED,
    ).cpu()

    continuation_ids = full[:, PREFIX_LEN:].tolist()
    hypothesis_texts = [sp.decode(seq) for seq in continuation_ids]
    generated_examples[name] = hypothesis_texts

    rows.append({
        "strategy": name,
        "BLEU-4": corpus_bleu4(reference_texts, hypothesis_texts),
        "chrF": corpus_chrf(reference_texts, hypothesis_texts),
        "ROUGE-L": rouge_l_f1(reference_texts, hypothesis_texts),
        "Distinct-1": distinct_n(continuation_ids, 1),
        "Distinct-2": distinct_n(continuation_ids, 2),
        "Repeated-3gram-rate": repetition_rate(continuation_ids, 3),
    })

generation_metrics_df = pd.DataFrame(rows)
display(generation_metrics_df)

print("\nHeld-out prefix:")
print(sp.decode(prefixes[0].tolist()))
print("\nReference continuation:")
print(reference_texts[0])

for name, _, _ in strategies:
    print(f"\n{name} continuation:")
    print(generated_examples[name][0])


### Attention Analysis

In [ ]:
@torch.inference_mode()
def capture_attention(eval_model, token_ids):
    was_training = eval_model.training
    eval_model.eval()
    token_ids = token_ids.to(device, dtype=torch.long)

    with autocast_context():
        logits, attentions = eval_model(token_ids, return_attentions=True)

    attentions = [a.float().cpu() for a in attentions]

    if was_training:
        eval_model.train()
    return logits.float().cpu(), attentions

ATTN_TOKENS = 32
attn_input = val_dataset[0][:ATTN_TOKENS].long().unsqueeze(0)
_, attention_layers = capture_attention(base_model, attn_input)

print("Layers captured:", len(attention_layers))
print("Layer-0 attention shape:", tuple(attention_layers[0].shape))

In [ ]:
def plot_attention_heatmaps(
    attention_layers,
    token_ids,
    tokenizer,
    layer_indices=(0, -1),
    head_indices=(0, 1, 2),
):
    pieces = [tokenizer.id_to_piece(int(i)) for i in token_ids]

    resolved_layers = [
        idx if idx >= 0 else len(attention_layers) + idx
        for idx in layer_indices
    ]

    for layer_idx in resolved_layers:
        for head_idx in head_indices:
            weights = attention_layers[layer_idx][0, head_idx].numpy()

            plt.figure(figsize=(8, 7))
            plt.imshow(weights, aspect="auto", interpolation="nearest")
            plt.colorbar(label="Attention weight")
            plt.title(f"Telugu Model H — layer {layer_idx}, head {head_idx}")
            plt.xlabel("Key position")
            plt.ylabel("Query position")

            if len(pieces) <= 40:
                ticks = np.arange(len(pieces))
                plt.xticks(ticks, pieces, rotation=90, fontsize=7)
                plt.yticks(ticks, pieces, fontsize=7)

            plt.tight_layout()
            plt.show()

plot_attention_heatmaps(
    attention_layers,
    attn_input[0].tolist(),
    sp,
    layer_indices=(0, config["n_layers"] - 1),
    head_indices=(0, 1, 2),
)

In [ ]:
def summarize_attention(attention_layers):
    rows = []

    for layer_idx, attn in enumerate(attention_layers):
        p = attn.clamp_min(1e-12)
        entropy = -(p * p.log()).sum(dim=-1).mean(dim=(0, 2))

        T = attn.size(-1)
        q_pos = torch.arange(T).view(1, 1, T, 1)
        k_pos = torch.arange(T).view(1, 1, 1, T)
        distance = (q_pos - k_pos).abs().float()

        mean_distance = (attn * distance).sum(dim=-1).mean(dim=(0, 2))

        for head_idx in range(attn.size(1)):
            rows.append({
                "layer": layer_idx,
                "head": head_idx,
                "attention_entropy": float(entropy[head_idx]),
                "mean_attention_distance": float(mean_distance[head_idx]),
            })

    return pd.DataFrame(rows)

attention_summary_df = summarize_attention(attention_layers)
display(attention_summary_df)

In [ ]:
def plot_attention_summary(summary_df):
    layer_summary = summary_df.groupby("layer", as_index=False).mean(numeric_only=True)

    plt.figure(figsize=(9, 4))
    plt.plot(
        layer_summary["layer"],
        layer_summary["attention_entropy"],
        marker="o",
        label="Mean entropy",
    )
    plt.title("Telugu Model H — mean attention entropy by layer")
    plt.xlabel("Layer")
    plt.ylabel("Attention entropy (nats)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.plot(
        layer_summary["layer"],
        layer_summary["mean_attention_distance"],
        marker="o",
        label="Mean attention distance",
    )
    plt.title("Telugu Model H — mean attention distance by layer")
    plt.xlabel("Layer")
    plt.ylabel("Mean |query-key| distance (tokens)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_attention_summary(attention_summary_df)